# Beads storage growth and automatic maintenance

Investigation of `crates/spur-pm/src/beads_crate` on 2026-10-27. Measure live files without deleting or changing issue data; inspect the exact pinned `beads_rust` source and current working tree; distinguish active data, JSONL backup history, WAL, temporary files, and recovery quarantine. Generated and executed through Notebook MCP.

The initial knowledge pack reported different code/analyst hashes. The later direct notebook attachment matched the exact graph hash; see the freshness cell. Use current source bytes for behavior claims. `snapshot.rs` is an in-memory optimistic concurrency token, not a disk snapshot writer. Tracking issue: `bd-2md`.

In [2]:
from pathlib import Path
from datetime import datetime, timezone
import json, sqlite3
roots = [Path('/Users/kevintruong/project/SpurCode/.beads'), Path('/Users/kevintruong/project/tcinvest_rest/.beads')]
beads_files = []
scan_errors = []
for root in roots:
    if not root.exists():
        continue
    for path in root.rglob('*'):
        try:
            if path.is_symlink() or not path.is_file():
                continue
            st = path.stat()
            rel = str(path.relative_to(root))
            category = ('history' if rel.startswith('.br_history/') else 'quarantine' if rel.startswith('quarantine/') else 'wal' if rel.endswith('-wal') else 'live_database' if rel == 'beads.db' else 'live_export' if rel == 'issues.jsonl' else 'temporary' if rel.endswith('.tmp') else 'other')
            beads_files.append(dict(project=root.parent.name, root=str(root), path=rel, category=category, bytes=st.st_size, modified_utc=datetime.fromtimestamp(st.st_mtime, timezone.utc).isoformat()))
        except OSError as error:
            scan_errors.append(dict(path=str(path), error=str(error)))
print(json.dumps({'observed_at_utc':datetime.now(timezone.utc).isoformat(),'file_count':len(beads_files),'scan_errors':scan_errors,'largest':sorted(beads_files,key=lambda x:x['bytes'],reverse=True)[:20]},indent=2))
inventory_path=Path('/Users/kevintruong/project/SpurCode/docs/research/2026-10-27-beads-storage-inventory.json')
inventory_path.write_text(json.dumps(beads_files, indent=2)+'\n')
print('Inventory saved:',inventory_path)


{
  "observed_at_utc": "2026-10-27T04:05:02.391568+00:00",
  "file_count": 6,
  "scan_errors": [],
  "largest": [
    {
      "project": "SpurCode",
      "root": "/Users/kevintruong/project/SpurCode/.beads",
      "path": "beads.db",
      "category": "live_database",
      "bytes": 364544,
      "modified_utc": "2026-10-27T04:04:51.907727+00:00"
    },
    {
      "project": "SpurCode",
      "root": "/Users/kevintruong/project/SpurCode/.beads",
      "path": "beads.db-wal",
      "category": "wal",
      "bytes": 115392,
      "modified_utc": "2026-10-27T04:04:51.906875+00:00"
    },
    {
      "project": "SpurCode",
      "root": "/Users/kevintruong/project/SpurCode/.beads",
      "path": "beads.db-shm",
      "category": "other",
      "bytes": 32768,
      "modified_utc": "2026-10-27T01:47:45.419770+00:00"
    },
    {
      "project": "SpurCode",
      "root": "/Users/kevintruong/project/SpurCode/.beads",
      "path": "issues.jsonl",
      "category": "live_export",
      "byt

In [3]:
-- Discover the inventory schema before aggregation.
DESCRIBE SELECT * FROM read_json_auto('/Users/kevintruong/project/SpurCode/docs/research/2026-10-27-beads-storage-inventory.json');

column_name,column_type,null,key,default,extra
project,VARCHAR,YES,NaN,NaN,NaN
root,VARCHAR,YES,NaN,NaN,NaN
path,VARCHAR,YES,NaN,NaN,NaN
category,VARCHAR,YES,NaN,NaN,NaN
bytes,BIGINT,YES,NaN,NaN,NaN
modified_utc,TIMESTAMP,YES,NaN,NaN,NaN


In [4]:
from pathlib import Path
import sqlite3, json
root = Path('/Users/kevintruong/project/SpurCode/.beads')
conn = sqlite3.connect((root/'beads.db').as_uri()+'?mode=ro', uri=True, timeout=1)
conn.execute('PRAGMA query_only = ON')
conn.execute('BEGIN')
schema = {name:[row[1] for row in conn.execute('PRAGMA table_info("'+name.replace('"','""')+'")')] for (name,) in conn.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")}
print('Schema:',json.dumps(schema,indent=2))
health = {key:conn.execute('PRAGMA '+key).fetchone()[0] for key in ['page_size','page_count','freelist_count','journal_mode','auto_vacuum']}
health['free_bytes'] = health['page_size']*health['freelist_count']
if 'issues' in schema and 'status' in schema['issues']:
    health['issues_by_status'] = conn.execute('SELECT status, count(*) FROM issues GROUP BY status ORDER BY status').fetchall()
if 'dirty_issues' in schema:
    health['dirty_issues'] = conn.execute('SELECT count(*) FROM dirty_issues').fetchone()[0]
for table in ['events','comments','dependencies','labels']:
    if table in schema:
        health[table+'_count'] = conn.execute('SELECT count(*) FROM '+table).fetchone()[0]
if 'metadata' in schema and {'key','value'}.issubset(schema['metadata']):
    health['flush_metadata'] = conn.execute("SELECT key,value FROM metadata WHERE key IN ('needs_flush','last_export_time','last_export_hash','jsonl_hash') ORDER BY key").fetchall()
conn.rollback()
conn.close()
print('Read-only health:',json.dumps(health,indent=2))


Schema: {
  "blocked_issues_cache": [
    "issue_id",
    "blocked_by",
    "blocked_at"
  ],
  "child_counters": [
    "parent_id",
    "last_child"
  ],
  "comments": [
    "id",
    "issue_id",
    "author",
    "text",
    "created_at"
  ],
  "config": [
    "key",
    "value"
  ],
  "dependencies": [
    "issue_id",
    "depends_on_id",
    "type",
    "created_at",
    "created_by",
    "metadata",
    "thread_id"
  ],
  "dirty_issues": [
    "issue_id",
    "marked_at"
  ],
  "events": [
    "id",
    "issue_id",
    "event_type",
    "actor",
    "old_value",
    "new_value",
    "comment",
    "created_at"
  ],
  "export_hashes": [
    "issue_id",
    "content_hash",
    "exported_at"
  ],
  "issues": [
    "id",
    "content_hash",
    "title",
    "description",
    "design",
    "acceptance_criteria",
    "notes",
    "status",
    "priority",
    "issue_type",
    "assignee",
    "owner",
    "estimated_minutes",
    "created_at",
    "created_by",
    "updated_at",
    "c

In [5]:
SELECT project, category, count(*) AS file_count, sum(bytes) AS total_bytes,
       round(sum(bytes)/1048576.0, 3) AS total_mib, max(bytes) AS largest_bytes
FROM read_json_auto('/Users/kevintruong/project/SpurCode/docs/research/2026-10-27-beads-storage-inventory.json')
GROUP BY project, category ORDER BY total_bytes DESC;

project,category,file_count,total_bytes,total_mib,largest_bytes
SpurCode,live_database,1,364544,0.348,364544
SpurCode,wal,1,115392,0.110,115392
SpurCode,other,2,32768,0.031,32768
SpurCode,live_export,1,7076,0.007,7076
SpurCode,history,1,0,0.000,0


In [6]:
ATTACH IF NOT EXISTS '/Users/kevintruong/project/SpurCode/.spur/analyst.duckdb' AS beads_code_20261027 (READ_ONLY);
SELECT table_name,column_name,data_type FROM information_schema.columns
WHERE table_catalog='beads_code_20261027' AND table_name IN ('_meta','nodes','edges','edges_unresolved')
ORDER BY table_name,ordinal_position;

table_name,column_name,data_type
_meta,artifact_dir,VARCHAR
_meta,graph_index_version,VARCHAR
_meta,schema_version,VARCHAR
_meta,manifest_version,VARCHAR
_meta,graph_content_hash,VARCHAR
_meta,extractor_version,VARCHAR
_meta,complete,BOOLEAN
_meta,node_count,BIGINT
_meta,resolved_edge_count,BIGINT
_meta,edges_by_dst_count,BIGINT


In [7]:
-- Model only: 100 distinct recent backups of equal size; no real files are deleted.
-- Proposed 256 MiB budget with two protected newest backups and existing max_count=100.
WITH scenarios(export_mib) AS (VALUES (1),(10),(50),(100),(500)),
policy AS (SELECT export_mib,100 AS existing_max_count,
                  greatest(2,least(100,floor(256.0/export_mib)::INTEGER)) AS proposed_count
           FROM scenarios)
SELECT export_mib, export_mib*existing_max_count AS existing_history_mib,
       proposed_count, export_mib*proposed_count AS proposed_history_mib,
       export_mib*proposed_count>256 AS protected_backups_exceed_budget,
       round(100.0*(1.0-proposed_count/100.0),1) AS reduction_pct
FROM policy ORDER BY export_mib;

export_mib,existing_history_mib,proposed_count,proposed_history_mib,protected_backups_exceed_budget,reduction_pct
1,100,100,100,False,0.0
10,1000,25,250,False,75.0
50,5000,5,250,False,95.0
100,10000,2,200,False,98.0
500,50000,2,1000,True,98.0


In [8]:
-- Freshness guard: graph MCP reported exact graph hash 8599245794571718b3153894d70d281c091e04ef3300eff4d66c0f3eb9fe6d59.
SELECT graph_content_hash,
       graph_content_hash='8599245794571718b3153894d70d281c091e04ef3300eff4d66c0f3eb9fe6d59' AS matches_exact_graph_at_investigation
FROM beads_code_20261027.main._meta;

graph_content_hash,matches_exact_graph_at_investigation
8599245794571718b3153894d70d281c091e04ef3300eff4d66c0f3eb9fe6d59,True


In [9]:
-- Indexed file inventory only; source verification is required while graph hashes differ.
SELECT file_path, count(*) AS indexed_symbols,
       count(*) FILTER (WHERE entity_name ILIKE '%flush%') AS flush_named_symbols,
       count(*) FILTER (WHERE entity_name ILIKE '%checkpoint%') AS checkpoint_named_symbols,
       count(*) FILTER (WHERE entity_name ILIKE '%maint%' OR entity_name ILIKE '%prune%' OR entity_name ILIKE '%retention%') AS maintenance_named_symbols
FROM beads_code_20261027.main.nodes
WHERE file_path LIKE 'crates/spur-pm/src/beads_crate/%'
GROUP BY file_path ORDER BY file_path;

file_path,indexed_symbols,flush_named_symbols,checkpoint_named_symbols,maintenance_named_symbols
crates/spur-pm/src/beads_crate/adapter.rs,44,2,1,0
crates/spur-pm/src/beads_crate/backoff.rs,15,0,0,0
crates/spur-pm/src/beads_crate/beads_advanced.rs,20,0,0,0
crates/spur-pm/src/beads_crate/beads_db.rs,43,0,5,0
crates/spur-pm/src/beads_crate/dependency_compat.rs,6,0,0,0
crates/spur-pm/src/beads_crate/init.rs,67,5,0,0
crates/spur-pm/src/beads_crate/issue_tracker.rs,49,0,0,0
crates/spur-pm/src/beads_crate/metrics.rs,33,6,2,0
crates/spur-pm/src/beads_crate/mod.rs,19,0,2,0
crates/spur-pm/src/beads_crate/reader_pool.rs,23,0,0,0


In [10]:
from pathlib import Path
import tempfile, sqlite3, json
# Controlled WAL experiment: temporary database only, with the writer kept open.
with tempfile.TemporaryDirectory(prefix='spur-beads-wal-analysis-') as tmp:
    db=Path(tmp)/'probe.db'
    writer=sqlite3.connect(db)
    writer.execute('PRAGMA journal_mode=WAL')
    writer.execute('PRAGMA wal_autocheckpoint=0')
    writer.execute('CREATE TABLE payload (data BLOB)')
    writer.commit()
    writer.executemany('INSERT INTO payload VALUES (?)',[(b'x'*4096,)]*256)
    writer.commit()
    wal=Path(str(db)+'-wal')
    before=wal.stat().st_size
    passive=writer.execute('PRAGMA wal_checkpoint(PASSIVE)').fetchone()
    after_passive=wal.stat().st_size
    truncate=writer.execute('PRAGMA wal_checkpoint(TRUNCATE)').fetchone()
    after_truncate=wal.stat().st_size
    print(json.dumps({'sqlite_version':sqlite3.sqlite_version,'wal_before_bytes':before,'passive_result':passive,'wal_after_passive_bytes':after_passive,'truncate_result':truncate,'wal_after_truncate_bytes':after_truncate},indent=2))
    assert before>0 and after_passive==before and after_truncate==0
    writer.close()


{
  "sqlite_version": "3.53.1",
  "wal_before_bytes": 1203072,
  "passive_result": [
    0,
    292,
    292
  ],
  "wal_after_passive_bytes": 1203072,
  "truncate_result": [
    0,
    0,
    0
  ],
  "wal_after_truncate_bytes": 0
}


## Findings and proposed maintenance

The measured checkout contains **519,780 bytes**, so large snapshot growth is not reproduced here. The pinned exporter already retains up to **100 backups / 30 days**, deduplicates unchanged content, and rotates even on the deduplication path. It has **no byte budget**. With 100 MiB full exports, 100 copies can occupy 10,000 MiB.

Current adapter writes request passive WAL checkpoints but do not schedule JSONL auto-flush. A read-only health check found two dirty issues and an older export timestamp. Quarantine moves recovery artifacts without reclaiming bytes. The temporary WAL experiment retained 1,203,072 bytes after PASSIVE and released them after TRUNCATE; it never touched the live database.

The initial knowledge-pack freshness mismatch did not persist in the direct notebook attachment: `.spur/analyst.duckdb` matched the exact graph hash observed during orientation. Source inspection remains authoritative because heuristic call edges confuse the adapter and dependency functions sharing the name `auto_flush`.

**Proposed first increment:** coalesced 30-second flush scheduling; a 15-minute history sweep with a 256 MiB soft byte budget; protection of the newest two available copies during additional byte pruning; nonblocking WAL truncation above 64 MiB; reuse of strict aged-temp cleanup; quarantine size diagnostics with deletion disabled. Upstream age/count pruning can leave fewer than two backups. Protected copies can exceed the byte budget, and canonical issue data is not automatically reduced.

Foreground callers acquire `.write.lock` before queuing writer jobs, so actor maintenance must try the lock once and skip on contention to avoid lock inversion. Check deadlines between jobs as well as while idle.

See the [full design and acceptance cases](../superpowers/specs/2026-10-27-beads-automatic-maintenance-design.md). All nine code cells ran successfully. This is investigation/design only; no Rust implementation or live cleanup was performed. Tracking issue: `bd-2md`.

## Approved implementation: TDD and pre/post solver evaluation

Implementation tracking: `bd-28k`, with retention `bd-2b6`, actor `bd-eqc`, and evaluation `bd-22u` in dependency order. The user approved the design and requested test-first development and pre/post solve.

Pre-solve receipts: approved policy defaults are feasible (`sol_f3b6ef78501d4fdb`, sat); resource aggregate capacity verifies that two 100 MiB backups fit 256 MiB (`sol_1dc164b074404398`, pass); an oldest-first four-entry model has no counterexample deleting protected entries or leaving removable excess (`sol_5043eab2920b428f`, unsat). File sizes range from zero to 64 GiB and protection from one to four entries. This bounded arithmetic proof does not prove filesystem or concurrency safety; runtime regressions cover those paths.

The post-implementation section will read a JSON receipt emitted by actual Rust APIs through the repository's remote build wrapper.